# NB2 — Train wav2vec2-BERT + CTC phoneme recognizer

Fine-tunes `facebook/w2v-bert-2.0` with a CTC head over the phoneme vocab from NB1.
Output is a **phoneme recognizer**: audio -> per-frame phoneme posteriors -> phoneme string.
No text decoder, so it does NOT autocorrect mispronunciations back to canonical text.

**Hardware:** fits Kaggle **GPU T4 x2** (or any 16 GB+ GPU) in fp16. ~600M params.

**Two-stage option (AraS2P recipe, higher shadda accuracy):** this notebook does the
core fine-tune on Iqra'Eval. The optional Stage-1 continue-pretrain on large Arabic
speech is described at the bottom and only matters if you want to push accuracy and
have access to a bigger GPU for a few hours.

In [ ]:
!pip install -q "transformers==4.44.2" "datasets==2.21.0" "accelerate==0.34.2" jiwer soundfile librosa
# Silence the per-step torch checkpoint UserWarning. Over a multi-hour run it is
# printed thousands of times into the captured notebook output, bloating the
# saved .ipynb until Kaggle's disk fills ("[Errno 28] No space left on device").
import warnings
warnings.filterwarnings("ignore", message=".*use_reentrant.*")
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
import json, os, time
import numpy as np
import torch
from dataclasses import dataclass
from typing import Dict, List, Union
from datasets import load_dataset, load_from_disk, Audio, concatenate_datasets

# ---- Single-GPU guard (avoids a DataParallel crash) ---------------------
# On Kaggle's 2x T4 the HF Trainer uses nn.DataParallel, which tries to GATHER
# each GPU's outputs back together. CTC outputs have VARIABLE sequence length
# per batch, so the two GPUs produce different-shaped tensors and gather fails.
# Simplest robust fix: use ONE GPU (DataParallel only activates with 2+).
if torch.cuda.device_count() > 1:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    print("Multiple GPUs detected -> restricting to GPU 0 to avoid DataParallel gather bug.")
print("visible GPUs:", torch.cuda.device_count())

# ---- HF auth + robust downloads -----------------------------------------
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = _tok
    os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
    print("HF_TOKEN loaded from Kaggle secrets.")
except Exception as e:
    print("No HF_TOKEN secret (anonymous, throttled downloads):", e)

os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")

def robust_load_dataset(repo, split, retries=6, wait=20):
    """load_dataset with retry/backoff so a transient 503/timeout on one shard
    doesn't kill the run. HF caches completed shards, so retries resume."""
    for i in range(retries):
        try:
            return load_dataset(repo, split=split)
        except Exception as e:
            print(f"  load {repo}[{split}] attempt {i+1}/{retries} failed: "
                  f"{type(e).__name__}: {str(e)[:160]}")
            if i == retries - 1:
                raise
            time.sleep(wait * (i + 1))

# ---- PATHS / CONFIG -----------------------------------------------------
MODEL_NAME = "facebook/w2v-bert-2.0"
SAMPLING_RATE = 16_000

TRAIN_REPO = "IqraEval/Iqra_train"
TTS_REPO   = "IqraEval/Iqra_TTS"
USE_TTS    = True
PHONEME_COL = "phoneme_ref"

# Paths into the attached `iqra-cache` Kaggle Dataset (verify with the !ls below).
# Set the two DISK paths to None to stream from HF instead.
CACHE_ROOT = "/kaggle/input/datasets/qasimmaajid/iqra-cache/iqra_prep"
LOCAL_TRAIN_DISK = f"{CACHE_ROOT}/iqra_train_arrow"
LOCAL_TTS_DISK   = f"{CACHE_ROOT}/iqra_tts_arrow"
VOCAB_PATH       = f"{CACHE_ROOT}/phoneme_vocab.json"

# ---- SMOKE TEST -----------------------------------------------------------
# Smoke run validated: PER 0.122 at 200 steps. Two prior full runs trained fine
# for ~4h then died on disk during a save. Fixes applied: warning-spam silenced,
# disable_tqdm, single safetensors checkpoint, use_reentrant set. 5000 steps
# (~5h) stays comfortably under both the disk and 12h limits.
SMOKE_TEST = False
if SMOKE_TEST:
    USE_TTS  = False
    MAX_STEPS = 200
    EVAL_STEPS = 100
    SUBSET_TRAIN = 512
    SUBSET_DEV   = 128
else:
    MAX_STEPS  = 5000    # ~5h on one T4; safe on disk + the 12h cap
    EVAL_STEPS = 1000
    SUBSET_TRAIN = None   # use all (~71k real + ~55k TTS)
    SUBSET_DEV   = None

OUTPUT_DIR = "/kaggle/working/w2vbert-quran-phoneme"
os.makedirs(OUTPUT_DIR, exist_ok=True)
MAX_AUDIO_S = 20.0

print("SMOKE_TEST:", SMOKE_TEST, "| MAX_STEPS:", MAX_STEPS, "| USE_TTS:", USE_TTS)
print("train cache:", LOCAL_TRAIN_DISK)
print("vocab exists:", os.path.exists(VOCAB_PATH))

In [ ]:
from transformers import AutoFeatureExtractor, Wav2Vec2BertForCTC, Wav2Vec2CTCTokenizer

with open(VOCAB_PATH, encoding="utf-8") as f:
    vocab = json.load(f)
print("vocab size:", len(vocab))

# Save vocab next to the model so the tokenizer can load it, then build a
# Wav2Vec2CTCTokenizer over space-separated phoneme symbols.
with open(os.path.join(OUTPUT_DIR, "vocab.json"), "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False)

tokenizer = Wav2Vec2CTCTokenizer(
    os.path.join(OUTPUT_DIR, "vocab.json"),
    unk_token="<unk>", pad_token="<pad>", word_delimiter_token=" ",
)
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
print("blank/pad id (CTC blank):", tokenizer.pad_token_id)

In [ ]:
# ---- Load + merge datasets ---------------------------------------------
# CRITICAL: the label must match what the AUDIO actually contains.
#   * Iqra_train  : clean recitation  -> target = phoneme_ref
#   * Iqra_TTS    : MISPRONOUNCED audio -> target = phoneme_mis  (NOT phoneme_ref!)
# Using phoneme_ref for the TTS audio would teach the model to hear phonemes that
# aren't there. We rename whichever column is correct to a common "labels_str".
if LOCAL_TRAIN_DISK:
    full = load_from_disk(LOCAL_TRAIN_DISK)    # DatasetDict with train/dev, no network
    train, dev = full["train"], full["dev"]
    print("loaded train+dev from local cache:", LOCAL_TRAIN_DISK)
else:
    train = robust_load_dataset(TRAIN_REPO, "train")
    dev   = robust_load_dataset(TRAIN_REPO, "dev")

def normalise(ds, label_col):
    keep = {"audio", label_col}
    drop = [c for c in ds.column_names if c not in keep]
    ds = ds.remove_columns(drop)
    if label_col != "labels_str":
        ds = ds.rename_column(label_col, "labels_str")
    return ds.cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))

train = normalise(train, PHONEME_COL)   # phoneme_ref for clean real audio
dev   = normalise(dev, PHONEME_COL)

if USE_TTS:
    try:
        if LOCAL_TTS_DISK:
            tts_full = load_from_disk(LOCAL_TTS_DISK)
            tts = tts_full["train"] if "train" in getattr(tts_full, "keys", lambda: [])() else tts_full
            print("loaded TTS from local cache:", LOCAL_TTS_DISK)
        else:
            tts = robust_load_dataset(TTS_REPO, "train")
        # phoneme_mis describes the actual (mispronounced) TTS audio.
        tts_col = "phoneme_mis" if "phoneme_mis" in tts.column_names else PHONEME_COL
        print("TTS target column:", tts_col)
        tts = normalise(tts, tts_col)
        train = concatenate_datasets([train, tts])
        print("merged TTS; train rows:", len(train))
    except Exception as e:
        print("TTS skipped:", e)

# Smoke test: take a tiny shuffled subset so a sanity run is minutes, not hours.
# keep_in_memory=True is REQUIRED when loading from /kaggle/input (read-only):
# .shuffle() otherwise writes its index temp file next to the source data and
# fails with "[Errno 30] Read-only file system". In-memory shuffle avoids that.
if SUBSET_TRAIN is not None:
    n = min(SUBSET_TRAIN, len(train))
    train = train.shuffle(seed=0, keep_in_memory=True).select(range(n))
if SUBSET_DEV is not None:
    n = min(SUBSET_DEV, len(dev))
    dev = dev.shuffle(seed=0, keep_in_memory=True).select(range(n))

print("train:", len(train), "| dev:", len(dev))

In [ ]:
# ---- On-the-fly preprocessing (no pre-materialised features) ------------
# .with_transform is lazy and RAM-safe: it decodes audio + builds features only
# for the rows in each batch, and writes NO cache (avoids the read-only-cache and
# RAM-OOM traps from the old pipeline). Iqra clips are <=22 s so no length filter
# is needed for a first run.
unk_id = tokenizer.convert_tokens_to_ids("<unk>")

def encode_labels(phoneme_str):
    return [vocab.get(t, unk_id) for t in str(phoneme_str).split() if t]

def make_transform(b):
    return {
        "input_features": [
            feature_extractor(a["array"], sampling_rate=SAMPLING_RATE).input_features[0]
            for a in b["audio"]
        ],
        "labels": [encode_labels(s) for s in b["labels_str"]],
    }

train_proc = train.with_transform(make_transform)
dev_proc   = dev.with_transform(make_transform)
print("lazy transforms ready")

In [ ]:
@dataclass
class CTCCollator:
    feature_extractor: object
    tokenizer: object
    def __call__(self, features):
        input_feats = [{"input_features": f["input_features"]} for f in features]
        batch = self.feature_extractor.pad(input_feats, return_tensors="pt")
        label_feats = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.tokenizer.pad(label_feats, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

collator = CTCCollator(feature_extractor, tokenizer)

In [ ]:
import jiwer

id2tok = {v: k for k, v in vocab.items()}

def decode_ids(ids):
    # CTC greedy collapse: drop repeats and blanks.
    out, prev = [], None
    blank = vocab["<pad>"]
    for i in ids:
        if i != prev and i != blank:
            out.append(id2tok.get(int(i), "<unk>"))
        prev = i
    return " ".join(out)

def compute_metrics(pred):
    logits = pred.predictions
    pred_ids = np.argmax(logits, axis=-1)
    labels = pred.label_ids
    labels = np.where(labels == -100, vocab["<pad>"], labels)
    pred_str  = [decode_ids(p) for p in pred_ids]
    label_str = [decode_ids(l) for l in labels]
    # Phoneme Error Rate = WER over space-separated phoneme tokens.
    per = jiwer.wer(label_str, pred_str)
    return {"per": per}

In [ ]:
from transformers import Wav2Vec2BertForCTC, TrainingArguments, Trainer, EarlyStoppingCallback

model = Wav2Vec2BertForCTC.from_pretrained(
    MODEL_NAME,
    vocab_size=len(vocab),
    pad_token_id=tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
    add_adapter=True,            # w2v-bert-2.0 CTC head needs the adapter
)
model.config.ctc_zero_infinity = True

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    warmup_steps=0 if SMOKE_TEST else 500,
    max_steps=MAX_STEPS,
    fp16=True,
    gradient_checkpointing=True,
    # Pass use_reentrant explicitly. Without this, torch prints a UserWarning at
    # EVERY step; over hours that warning spam is captured into the notebook
    # output and bloats the saved .ipynb until disk fills ([Errno 28]).
    gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_steps=EVAL_STEPS,
    # DISK: Kaggle /kaggle/working ~20 GB. Keep only the single best checkpoint,
    # use safetensors. (The log-bloat fix above is the other half of the cure.)
    save_total_limit=1,
    save_safetensors=True,
    logging_steps=100 if SMOKE_TEST else 200,   # fewer log rows = smaller .ipynb
    disable_tqdm=True,                           # no progress-bar spam in the log
    load_best_model_at_end=not SMOKE_TEST,
    metric_for_best_model="per",
    greater_is_better=False,
    dataloader_num_workers=2,
    # REQUIRED: we use .with_transform, so input_features/labels are produced
    # lazily and don't appear in dataset.column_names. Without this, the Trainer
    # strips every column and errors with "No columns match forward signature".
    remove_unused_columns=False,
    label_names=["labels"],
    report_to="none",
)

callbacks = [] if SMOKE_TEST else [EarlyStoppingCallback(early_stopping_patience=3)]

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_proc,
    eval_dataset=dev_proc,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=callbacks,
)
try:
    trainer.tokenizer = tokenizer  # for saving
except Exception:
    pass

In [ ]:
trainer.train()
trainer.save_model(os.path.join(OUTPUT_DIR, "final"))
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, "final"))
feature_extractor.save_pretrained(os.path.join(OUTPUT_DIR, "final"))
print("saved final model to", os.path.join(OUTPUT_DIR, "final"))

### Optional Stage-1 (accuracy push, AraS2P)
If you want the higher-accuracy 2-stage recipe and have a bigger GPU for a few hours:
1. Build speech->phoneme pairs from Common Voice (Ar) / SADA / MASC using the MSA
   phonetiser, then continue-pretrain this same CTC objective on that large set first.
2. Then run THIS notebook's fine-tune starting from that checkpoint (set `MODEL_NAME`
   to your Stage-1 output dir).
This is what closes most of the shadda gap; the fine-tune-only path above already
gives a solid model on free T4s.

### Output of NB2
- `w2vbert-quran-phoneme/final/` — model + tokenizer + feature extractor. Attach to NB3.